# RDF Extraction Evaluation V2

## New Approach: datatable + caption_L1 only

This notebook uses a **new gold TTL conversion** that:
- Uses **only** `datatable` and `caption_L1` fields from the original VisText JSON
- Ignores `scenegraph`, `caption_L2L3`, `L1_properties`, etc.
- Generates TTL matching the LLM prediction schema (`chart:category`, `chart:value`)

### Differences from V1
- V1 used the original `json_to_ttl_converter.py` which had issues with category name concatenation
- V2 uses a cleaner converter that matches the prediction schema exactly

---

In [33]:
import os
import json
import hashlib
from pathlib import Path
from typing import Optional, Set, Tuple, Dict, List
import pandas as pd
from rdflib import Graph, Literal, Namespace
from rdflib.namespace import RDF, RDFS

# =========================
# 1. CONFIGURATION
# =========================
WORK_DIR = os.getcwd()

# Source JSON files
GT_SOURCE_JSON = os.path.join(WORK_DIR, "labels_1000")

# NEW gold TTL folder (V2 conversion)
GT_DEST_TTL_V2 = os.path.join(WORK_DIR, "gold_ttl_1000")

# Prediction outputs (note: folder has space in name)
OUTPUT_ROOT = os.path.join(WORK_DIR, "output 1000")

# Directories containing LLM outputs
OUTPUT_SOURCES = [
    ("Zeroshot", os.path.join(OUTPUT_ROOT, "vistext_Zeroshot_outputs")),
    ("Oneshot",  os.path.join(OUTPUT_ROOT, "vistext_Oneshot_outputs")),
    ("Fewshot",  os.path.join(OUTPUT_ROOT, "vistext_Fewshot_outputs")),
]

print(f"Working directory: {WORK_DIR}")
print(f"Gold JSON source: {GT_SOURCE_JSON}")
print(f"Gold TTL V2 destination: {GT_DEST_TTL_V2}")
print(f"Predictions root: {OUTPUT_ROOT}")

Working directory: /Users/ali/Desktop/RA/Wageningen/Github/Uploading/V12/Soil-Health/VisText/Evaluation/نتیجه گرفتن/Inferance/Evaluate 1000
Gold JSON source: /Users/ali/Desktop/RA/Wageningen/Github/Uploading/V12/Soil-Health/VisText/Evaluation/نتیجه گرفتن/Inferance/Evaluate 1000/labels_1000
Gold TTL V2 destination: /Users/ali/Desktop/RA/Wageningen/Github/Uploading/V12/Soil-Health/VisText/Evaluation/نتیجه گرفتن/Inferance/Evaluate 1000/gold_ttl_1000
Predictions root: /Users/ali/Desktop/RA/Wageningen/Github/Uploading/V12/Soil-Health/VisText/Evaluation/نتیجه گرفتن/Inferance/Evaluate 1000/output 1000


## Step 1: Generate Gold TTL V2

Run the new converter to generate gold TTLs using only `datatable` + `caption_L1`.

## Step 0: Complete Empty JSON Files

Many JSON files in `labels_1000` are empty (contain only `[]`). We need to fill them using data from `data_train.json`.

In [21]:
# Complete empty JSON files in labels_1000 using data_train.json
import json
from pathlib import Path

# Load training data
print("Loading data_train.json...")
with open("data_train.json", 'r', encoding='utf-8') as f:
    train_data = json.load(f)

# Index by img_id
img_id_map = {}
for entry in train_data:
    if isinstance(entry, dict) and 'img_id' in entry:
        img_id = str(entry['img_id'])
        img_id_map[img_id] = entry

print(f"Loaded {len(img_id_map)} entries from training data\n")

# Find and fix empty JSON files
labels_dir = Path("labels_1000")
json_files = list(labels_dir.glob("*.json"))

filled_count = 0
not_found_count = 0
already_filled_count = 0

print(f"Scanning {len(json_files)} JSON files...")
for json_file in json_files:
    with open(json_file, 'r', encoding='utf-8') as f:
        content = json.load(f)
    
    # Check if it's an empty list
    if isinstance(content, list) and len(content) == 0:
        img_id = json_file.stem
        
        if img_id in img_id_map:
            # Fill the file with data
            with open(json_file, 'w', encoding='utf-8') as f:
                json.dump(img_id_map[img_id], f, indent=4, ensure_ascii=False)
            print(f"✓ Filled {json_file.name} with data for img_id={img_id}")
            filled_count += 1
        else:
            print(f"✗ No data found for img_id={img_id} ({json_file.name})")
            not_found_count += 1
    else:
        already_filled_count += 1

print(f"\n{'='*70}")
print(f"SUMMARY")
print(f"{'='*70}")
print(f"Total JSON files: {len(json_files)}")
print(f"Already filled: {already_filled_count}")
print(f"Successfully filled: {filled_count}")
print(f"Not found in training data: {not_found_count}")
print(f"\n✓ Operation complete!")

Loading data_train.json...
Loaded 7057 entries from training data

Scanning 1000 JSON files...

SUMMARY
Total JSON files: 1000
Already filled: 1000
Successfully filled: 0
Not found in training data: 0

✓ Operation complete!


In [23]:
# Run the V2 converter
!python3 json_to_ttl_converter_v2.py "labels_1000" -o "gold_ttl_1000"

Converted 1000 files, 0 errors

✓ Conversion complete!


In [22]:
# Quick summary of conversion results
import os
labels_count = len([f for f in os.listdir("labels_1000") if f.endswith('.json')])
ttl_count = len([f for f in os.listdir("gold_ttl_1000") if f.endswith('.ttl')])

print(f"JSON files in labels_1000: {labels_count}")
print(f"TTL files in gold_ttl_1000: {ttl_count}")
print(f"Conversion success rate: {ttl_count}/{labels_count} = {100*ttl_count/labels_count:.1f}%")

JSON files in labels_1000: 1000
TTL files in gold_ttl_1000: 1000
Conversion success rate: 1000/1000 = 100.0%


In [24]:
# Check which files failed to convert
json_files = set(Path("labels_1000").glob("*.json"))
ttl_files = set(Path("gold_ttl_1000").glob("*.ttl"))

json_ids = {f.stem for f in json_files}
ttl_ids = {f.stem for f in ttl_files}

missing_ttl = sorted(json_ids - ttl_ids)
print(f"Files that failed to convert ({len(missing_ttl)}):")
print(", ".join(missing_ttl[:20]))
if len(missing_ttl) > 20:
    print(f"... and {len(missing_ttl) - 20} more")

Files that failed to convert (0):



In [25]:
# Complete remaining empty files using data_validation.json
print("Loading data_validation.json...")
with open("data_validation.json", 'r', encoding='utf-8') as f:
    validation_data = json.load(f)

# Index by img_id
validation_map = {}
for entry in validation_data:
    if isinstance(entry, dict) and 'img_id' in entry:
        img_id = str(entry['img_id'])
        validation_map[img_id] = entry

print(f"Loaded {len(validation_map)} entries from validation data\n")

# Find remaining empty JSON files
labels_dir = Path("labels_1000")
filled_count = 0
not_found_count = 0

print(f"Scanning for remaining empty files...")
for json_file in labels_dir.glob("*.json"):
    with open(json_file, 'r', encoding='utf-8') as f:
        content = json.load(f)
    
    # Check if it's still an empty list
    if isinstance(content, list) and len(content) == 0:
        img_id = json_file.stem
        
        if img_id in validation_map:
            # Fill the file with validation data
            with open(json_file, 'w', encoding='utf-8') as f:
                json.dump(validation_map[img_id], f, indent=4, ensure_ascii=False)
            print(f"✓ Filled {json_file.name} with validation data for img_id={img_id}")
            filled_count += 1
        else:
            print(f"✗ No data found in validation set for img_id={img_id} ({json_file.name})")
            not_found_count += 1

print(f"\n{'='*70}")
print(f"SUMMARY (Validation Data)")
print(f"{'='*70}")
print(f"Successfully filled from validation: {filled_count}")
print(f"Still not found: {not_found_count}")
print(f"\n✓ Operation complete!")

Loading data_validation.json...
Loaded 883 entries from validation data

Scanning for remaining empty files...

SUMMARY (Validation Data)
Successfully filled from validation: 0
Still not found: 0

✓ Operation complete!


In [26]:
# Rerun the TTL converter now that all files are filled
!python3 json_to_ttl_converter_v2.py "labels_1000" -o "gold_ttl_1000"

Converted 1000 files, 0 errors

✓ Conversion complete!


In [27]:
# Final summary after completing all files
import os
labels_count = len([f for f in os.listdir("labels_1000") if f.endswith('.json')])
ttl_count = len([f for f in os.listdir("gold_ttl_1000") if f.endswith('.ttl')])

print("="*70)
print("FINAL CONVERSION SUMMARY")
print("="*70)
print(f"JSON files in labels_1000: {labels_count}")
print(f"TTL files in gold_ttl_1000: {ttl_count}")
print(f"Conversion success rate: {ttl_count}/{labels_count} = {100*ttl_count/labels_count:.1f}%")

# Check for any remaining empty files
empty_count = 0
for json_file in Path("labels_1000").glob("*.json"):
    with open(json_file, 'r', encoding='utf-8') as f:
        content = json.load(f)
    if isinstance(content, list) and len(content) == 0:
        empty_count += 1

print(f"\nRemaining empty JSON files: {empty_count}")
print("="*70)

FINAL CONVERSION SUMMARY
JSON files in labels_1000: 1000
TTL files in gold_ttl_1000: 1000
Conversion success rate: 1000/1000 = 100.0%

Remaining empty JSON files: 0


In [16]:
# Complete remaining files using data_test.json
# Also fix files that are lists containing a single dict (should be just the dict)
print("Loading data_test.json...")
with open("data_test.json", 'r', encoding='utf-8') as f:
    test_data = json.load(f)

# Index by img_id
test_map = {}
for entry in test_data:
    if isinstance(entry, dict) and 'img_id' in entry:
        img_id = str(entry['img_id'])
        test_map[img_id] = entry

print(f"Loaded {len(test_map)} entries from test data\n")

# Process all JSON files
labels_dir = Path("labels_1000")
filled_empty = 0
fixed_list = 0
not_found_count = 0

print(f"Processing JSON files...")
for json_file in sorted(labels_dir.glob("*.json")):
    with open(json_file, 'r', encoding='utf-8') as f:
        content = json.load(f)
    
    needs_fix = False
    img_id = json_file.stem
    
    # Check if it's an empty list
    if isinstance(content, list) and len(content) == 0:
        needs_fix = True
        fix_type = "empty list"
    # Check if it's a list with a single dictionary (should be unwrapped)
    elif isinstance(content, list) and len(content) == 1 and isinstance(content[0], dict):
        needs_fix = True
        fix_type = "list wrapper"
        # Use the dict inside the list
        if img_id in test_map:
            content = test_map[img_id]
        else:
            # Just unwrap the existing content
            content = content[0]
    
    if needs_fix:
        if img_id in test_map:
            # Fill/fix with test data
            with open(json_file, 'w', encoding='utf-8') as f:
                json.dump(test_map[img_id], f, indent=4, ensure_ascii=False)
            
            if fix_type == "empty list":
                print(f"✓ Filled {json_file.name} with test data for img_id={img_id}")
                filled_empty += 1
            else:
                print(f"✓ Fixed {json_file.name} (unwrapped list) with test data for img_id={img_id}")
                fixed_list += 1
        elif fix_type == "list wrapper":
            # Just unwrap the list
            with open(json_file, 'w', encoding='utf-8') as f:
                json.dump(content, f, indent=4, ensure_ascii=False)
            print(f"✓ Fixed {json_file.name} (unwrapped list)")
            fixed_list += 1
        else:
            print(f"✗ No data found in test set for img_id={img_id} ({json_file.name})")
            not_found_count += 1

print(f"\n{'='*70}")
print(f"SUMMARY (Test Data)")
print(f"{'='*70}")
print(f"Filled empty files from test data: {filled_empty}")
print(f"Fixed list-wrapped files: {fixed_list}")
print(f"Still not found: {not_found_count}")
print(f"\n✓ Operation complete!")

Loading data_test.json...
Loaded 882 entries from test data

Processing JSON files...

SUMMARY (Test Data)
Filled empty files from test data: 0
Fixed list-wrapped files: 0
Still not found: 0

✓ Operation complete!


In [17]:
# Check sample file 1073.json structure
with open("labels_1000/1073.json", 'r') as f:
    sample = json.load(f)
print(f"Type: {type(sample)}")
if isinstance(sample, list):
    print(f"Length: {len(sample)}")
    if len(sample) > 0:
        print(f"First element type: {type(sample[0])}")
        if isinstance(sample[0], dict):
            print(f"Keys: {list(sample[0].keys())[:5]}")

Type: <class 'list'>
Length: 2
First element type: <class 'dict'>
Keys: ['caption_id', 'img_id', 'split', 'scenegraph', 'datatable']


In [18]:
# Fix files that are lists (multiple captions) - unwrap to use first caption
labels_dir = Path("labels_1000")
unwrapped_count = 0

print("Fixing list-wrapped JSON files...")
for json_file in sorted(labels_dir.glob("*.json")):
    with open(json_file, 'r', encoding='utf-8') as f:
        content = json.load(f)
    
    # If it's a list containing dict(s), unwrap it
    if isinstance(content, list) and len(content) > 0 and isinstance(content[0], dict):
        # Take the first caption/entry
        unwrapped = content[0]
        
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(unwrapped, f, indent=4, ensure_ascii=False)
        
        print(f"✓ Unwrapped {json_file.name} (was list with {len(content)} entries)")
        unwrapped_count += 1

print(f"\n{'='*70}")
print(f"Fixed {unwrapped_count} list-wrapped files")
print(f"{'='*70}")

Fixing list-wrapped JSON files...
✓ Unwrapped 1073.json (was list with 2 entries)
✓ Unwrapped 1168.json (was list with 2 entries)
✓ Unwrapped 1170.json (was list with 3 entries)
✓ Unwrapped 1231.json (was list with 4 entries)
✓ Unwrapped 1404.json (was list with 2 entries)
✓ Unwrapped 1439.json (was list with 2 entries)
✓ Unwrapped 1515.json (was list with 3 entries)
✓ Unwrapped 1566.json (was list with 3 entries)
✓ Unwrapped 1808.json (was list with 3 entries)
✓ Unwrapped 1907.json (was list with 2 entries)
✓ Unwrapped 2048.json (was list with 3 entries)
✓ Unwrapped 2094.json (was list with 2 entries)
✓ Unwrapped 2258.json (was list with 2 entries)
✓ Unwrapped 2392.json (was list with 3 entries)
✓ Unwrapped 2465.json (was list with 3 entries)
✓ Unwrapped 2531.json (was list with 2 entries)
✓ Unwrapped 3008.json (was list with 2 entries)
✓ Unwrapped 3062.json (was list with 2 entries)
✓ Unwrapped 3120.json (was list with 3 entries)
✓ Unwrapped 315.json (was list with 2 entries)
✓ Unwra

In [19]:
# Rerun the TTL converter after unwrapping list files
!python3 json_to_ttl_converter_v2.py "labels_1000" -o "gold_ttl_1000"

Converted 1000 files, 0 errors

✓ Conversion complete!


In [20]:
# 🎉 FINAL SUMMARY - All files successfully converted!
import os
labels_count = len([f for f in os.listdir("labels_1000") if f.endswith('.json')])
ttl_count = len([f for f in os.listdir("gold_ttl_1000") if f.endswith('.ttl')])

print("="*70)
print("🎉 FINAL CONVERSION SUMMARY - SUCCESS!")
print("="*70)
print(f"JSON files in labels_1000: {labels_count}")
print(f"TTL files in gold_ttl_1000: {ttl_count}")
print(f"Conversion success rate: {ttl_count}/{labels_count} = {100*ttl_count/labels_count:.1f}%")

# Verify no empty or list-wrapped files remain
empty_count = 0
list_count = 0
for json_file in Path("labels_1000").glob("*.json"):
    with open(json_file, 'r', encoding='utf-8') as f:
        content = json.load(f)
    if isinstance(content, list):
        if len(content) == 0:
            empty_count += 1
        else:
            list_count += 1

print(f"\nRemaining empty JSON files: {empty_count}")
print(f"Remaining list-wrapped files: {list_count}")
print("="*70)
print("\n✅ All JSON files have been properly formatted and converted to TTL!")

🎉 FINAL CONVERSION SUMMARY - SUCCESS!
JSON files in labels_1000: 1000
TTL files in gold_ttl_1000: 1000
Conversion success rate: 1000/1000 = 100.0%

Remaining empty JSON files: 0
Remaining list-wrapped files: 0

✅ All JSON files have been properly formatted and converted to TTL!


In [28]:
# Verify conversion
gold_files = list(Path(GT_DEST_TTL_V2).glob("*.ttl"))
print(f"Generated {len(gold_files)} gold TTL files")

# Show sample
if gold_files:
    sample_file = sorted(gold_files)[0]
    print(f"\n--- Sample: {sample_file.name} ---")
    with open(sample_file) as f:
        print(f.read()[:2000])

Generated 1000 gold TTL files

--- Sample: 100.ttl ---
@prefix ex:    <http://example.org/> .
@prefix chart: <http://example.org/chart#> .
@prefix xsd:   <http://www.w3.org/2001/XMLSchema#> .

ex:chart-100 a chart:Chart ;
    chart:chartType "bar" ;
    chart:title "Annual number of hospital beds in the United Kingdom (UK) from 2000 to 2019" ;
    chart:xAxis ex:x-100 ;
    chart:yAxis ex:y-100 .

ex:x-100 a chart:Axis ;
    chart:label "Year" .

ex:y-100 a chart:Axis ;
    chart:label "" .

ex:series-100-1 a chart:Series .

ex:mark-100-1-1 a chart:DataPoint ;
    chart:series ex:series-100-1 ;
    chart:category "Number of beds 2019***" ;
    chart:value "163873"^^xsd:decimal .

ex:mark-100-1-2 a chart:DataPoint ;
    chart:series ex:series-100-1 ;
    chart:category "2018**" ;
    chart:value "165844"^^xsd:decimal .

ex:mark-100-1-3 a chart:DataPoint ;
    chart:series ex:series-100-1 ;
    chart:category "*" ;
    chart:value "183849"^^xsd:decimal .



## Step 2: Evaluation Functions

Same evaluation logic as V1, but with the fixed extraction function.

In [35]:
def load_ttl_graph(ttl_path: str) -> Optional[Graph]:
    """Load and parse a TTL file into an RDF graph."""
    g = Graph()
    try:
        g.parse(ttl_path, format="turtle")
        return g
    except Exception as e:
        return None

def normalize_string(s: str) -> str:
    """Normalize string for comparison."""
    return s.strip().lower()

def normalize_value(val: str) -> Optional[float]:
    """Convert string value to float."""
    try:
        return float(val.replace(',', ''))
    except:
        return None

def extract_chart_metadata(g: Graph) -> dict:
    """Extract chart metadata (title, type) from graph."""
    metadata = {'title': None, 'type': None}
    for s, p, o in g:
        pred_str = str(p).lower()
        if 'title' in pred_str and isinstance(o, Literal):
            metadata['title'] = normalize_string(str(o))
        if 'charttype' in pred_str.replace('_', '') and isinstance(o, Literal):
            metadata['type'] = normalize_string(str(o))
    return metadata

def extract_data_points(g: Graph) -> Set[Tuple[str, float]]:
    """
    Extract data points using chart:category / chart:value schema.
    Handles both # and / URI endings.
    """
    data_points = set()
    for s in g.subjects():
        category = None
        value = None
        for pred, obj in g.predicate_objects(s):
            pred_str = str(pred)
            # Check for category predicate
            if pred_str.endswith('#category') or pred_str.endswith('/category'):
                if isinstance(obj, Literal):
                    category = normalize_string(str(obj))
            # Check for value predicate
            if pred_str.endswith('#value') or pred_str.endswith('/value'):
                if isinstance(obj, Literal):
                    value = normalize_value(str(obj))
        
        if category and value is not None:
            data_points.add((category, value))
    return data_points

def compare_data_points(gt_points: set, out_points: set, tolerance_pct: float = 0.05) -> int:
    """
    Compare data points with tolerance-based matching.
    Returns number of matched points.
    """
    matched = 0
    used_out = set()
    
    for gt_cat, gt_val in sorted(list(gt_points)):
        best_match = None
        min_error = float('inf')
        
        for out_pt in out_points:
            if out_pt in used_out:
                continue
            
            out_cat, out_val = out_pt
            
            # Category must match exactly (normalized)
            if gt_cat == out_cat:
                # Calculate relative error
                if gt_val != 0:
                    rel_error = abs(gt_val - out_val) / abs(gt_val)
                elif abs(gt_val - out_val) > 1e-6:
                    rel_error = 1.0
                else:
                    rel_error = 0.0

                if rel_error <= tolerance_pct:
                    if rel_error < min_error:
                        min_error = rel_error
                        best_match = out_pt

        if best_match:
            matched += 1
            used_out.add(best_match)
            
    return matched

print("Evaluation functions defined.")

Evaluation functions defined.


In [36]:
def evaluate_outputs(gt_dir: str, out_dir: str, verbose: bool = False) -> dict:
    """
    Evaluate predictions against gold standard.
    Returns dictionary of metrics.
    """
    gt_files = {Path(p).stem: str(p) for p in Path(gt_dir).glob("*.ttl")}
    out_files = {Path(p).stem: str(p) for p in Path(out_dir).glob("*.ttl")}
    common_ids = sorted(set(gt_files.keys()) & set(out_files.keys()))
    
    if verbose:
        print(f"Gold files: {len(gt_files)}")
        print(f"Prediction files: {len(out_files)}")
        print(f"Common files: {len(common_ids)}")
    
    macro_p, macro_r, macro_f1, macro_j = [], [], [], []
    relaxed_f1_10, relaxed_f1_15, relaxed_f1_20 = [], [], []
    
    parse_errors = 0
    empty_graphs = 0
    
    per_file_details = []
    
    for img_id in common_ids:
        gt_graph = load_ttl_graph(gt_files[img_id])
        out_graph = load_ttl_graph(out_files[img_id])
        
        if gt_graph is None:
            continue
        if out_graph is None: 
            parse_errors += 1
            macro_p.append(0); macro_r.append(0); macro_f1.append(0); macro_j.append(0)
            relaxed_f1_10.append(0); relaxed_f1_15.append(0); relaxed_f1_20.append(0)
            continue
            
        if len(list(out_graph)) == 0:
            empty_graphs += 1

        gt_points = extract_data_points(gt_graph)
        out_points = extract_data_points(out_graph)
        
        # Different tolerance levels
        matched_5 = compare_data_points(gt_points, out_points, tolerance_pct=0.05)
        matched_10 = compare_data_points(gt_points, out_points, tolerance_pct=0.10)
        matched_15 = compare_data_points(gt_points, out_points, tolerance_pct=0.15)
        matched_20 = compare_data_points(gt_points, out_points, tolerance_pct=0.20)
        
        # Metadata match
        gt_meta = extract_chart_metadata(gt_graph)
        out_meta = extract_chart_metadata(out_graph)
        
        meta_matches = 0
        meta_total = 0
        if gt_meta['title'] and out_meta['title']:
            meta_total += 1
            if gt_meta['title'] == out_meta['title']:
                meta_matches += 1
        if gt_meta['type'] and out_meta['type']:
            meta_total += 1
            if gt_meta['type'] == out_meta['type']:
                meta_matches += 1

        # Totals for metrics
        total_gt = len(gt_points) + meta_total
        total_out = len(out_points) + meta_total
        
        # Calculate metrics at different tolerances
        def calc_metrics(matched, total_gt, total_out, meta_matches):
            total_matched = matched + meta_matches
            p = total_matched / total_out if total_out else 0
            r = total_matched / total_gt if total_gt else 0
            f1 = 2 * p * r / (p + r) if (p + r) else 0
            j = total_matched / (total_gt + total_out - total_matched) if (total_gt + total_out - total_matched) else 0
            return p, r, f1, j
        
        p, r, f1, j = calc_metrics(matched_5, total_gt, total_out, meta_matches)
        _, _, f1_10, _ = calc_metrics(matched_10, total_gt, total_out, meta_matches)
        _, _, f1_15, _ = calc_metrics(matched_15, total_gt, total_out, meta_matches)
        _, _, f1_20, _ = calc_metrics(matched_20, total_gt, total_out, meta_matches)
        
        macro_p.append(p)
        macro_r.append(r)
        macro_f1.append(f1)
        macro_j.append(j)
        relaxed_f1_10.append(f1_10)
        relaxed_f1_15.append(f1_15)
        relaxed_f1_20.append(f1_20)
        
        per_file_details.append({
            'id': img_id,
            'gt_points': len(gt_points),
            'pred_points': len(out_points),
            'matched_5': matched_5,
            'matched_15': matched_15,
            'f1': f1
        })
        
    def avg(l): return sum(l)/len(l) if l else 0
    
    return {
        "Precision": avg(macro_p),
        "Recall": avg(macro_r),
        "F1 (5%)": avg(macro_f1),
        "Jaccard": avg(macro_j),
        "F1 (10%)": avg(relaxed_f1_10),
        "F1 (15%)": avg(relaxed_f1_15),
        "F1 (20%)": avg(relaxed_f1_20),
        "Parse Errors": parse_errors,
        "Empty Graphs": empty_graphs,
        "_details": per_file_details
    }

print("evaluate_outputs function defined.")

evaluate_outputs function defined.


## Step 3: Run Evaluation

In [37]:
print("Running Evaluation V2...")
print(f"Gold TTL folder: {GT_DEST_TTL_V2}")
print()

results = []
details_all = {}

for name, path in OUTPUT_SOURCES:
    print(f"Evaluating: {name}...")
    if os.path.exists(path):
        metrics = evaluate_outputs(GT_DEST_TTL_V2, path, verbose=False)
        details_all[name] = metrics.pop('_details')
        metrics['Method'] = name
        results.append(metrics)
    else:
        print(f"  Missing directory: {path}")

# Debug: Check if results is empty
if not results:
    print("\n❌ ERROR: No results found!")
    print("Please check that OUTPUT_SOURCES directories exist:")
    for name, path in OUTPUT_SOURCES:
        exists = "✓" if os.path.exists(path) else "✗"
        print(f"  {exists} {name}: {path}")
else:
    df = pd.DataFrame(results)
    
    # Debug: Show what columns we have
    print(f"\nDataFrame has {len(df)} rows and columns: {list(df.columns)}\n")
    
    # Reorder columns (only use columns that exist)
    desired_cols = ['Method', 'Precision', 'Recall', 'F1 (5%)', 'F1 (10%)', 'F1 (15%)', 'F1 (20%)', 'Jaccard', 'Parse Errors', 'Empty Graphs']
    available_cols = [col for col in desired_cols if col in df.columns]
    
    if available_cols:
        df = df[available_cols]
        
        print("=" * 80)
        print("EVALUATION V2 RESULTS (Gold: datatable + caption_L1 only)")
        print("=" * 80)
        print(df.to_string(index=False))
    else:
        print("\n❌ ERROR: DataFrame columns don't match expected format!")
        print(f"Expected: {desired_cols}")
        print(f"Got: {list(df.columns)}")

Running Evaluation V2...
Gold TTL folder: /Users/ali/Desktop/RA/Wageningen/Github/Uploading/V12/Soil-Health/VisText/Evaluation/نتیجه گرفتن/Inferance/Evaluate 1000/gold_ttl_1000

Evaluating: Zeroshot...
Evaluating: Oneshot...
Evaluating: Fewshot...

DataFrame has 3 rows and columns: ['Precision', 'Recall', 'F1 (5%)', 'Jaccard', 'F1 (10%)', 'F1 (15%)', 'F1 (20%)', 'Parse Errors', 'Empty Graphs', 'Method']

EVALUATION V2 RESULTS (Gold: datatable + caption_L1 only)
  Method  Precision   Recall  F1 (5%)  F1 (10%)  F1 (15%)  F1 (20%)  Jaccard  Parse Errors  Empty Graphs
Zeroshot   0.361391 0.240525 0.270752  0.292099  0.306132  0.315316 0.197314           431             0
 Oneshot   0.883000 0.211158 0.316327  0.316327  0.316327  0.316327 0.211158            18             0
 Fewshot   0.642020 0.429044 0.480345  0.517965  0.540357  0.556320 0.355580            51             0


## Step 4: Detailed Analysis

In [38]:
# Show per-file breakdown for Zeroshot
print("Per-file breakdown (Zeroshot, first 20 files):")
print()
print(f"{'ID':<10} {'GT Pts':>8} {'Pred Pts':>10} {'Match@5%':>10} {'Match@15%':>11} {'F1':>8}")
print("-" * 60)

for d in details_all.get('Zeroshot', [])[:20]:
    print(f"{d['id']:<10} {d['gt_points']:>8} {d['pred_points']:>10} {d['matched_5']:>10} {d['matched_15']:>11} {d['f1']:>8.3f}")

Per-file breakdown (Zeroshot, first 20 files):

ID           GT Pts   Pred Pts   Match@5%   Match@15%       F1
------------------------------------------------------------
100               3         20          0           1    0.148
1033              1          0          0           0    0.800
1046             20         20          4           8    0.227
1052              2          0          0           0    0.667
1068             10         10          1           1    0.167
1075              2         19          0           0    0.160
1077             11         11         10          10    0.923
1088             11         10          2           2    0.320
1093             18         17         10          14    0.615
1133              1          0          0           0    0.667
1137              2          0          0           0    0.667
1143              3          0          0           0    0.571
1150             72          0          0           0    0.053
1159     

In [39]:
# Sample comparison: Gold vs Prediction
print("="*70)
print("Sample comparison: Gold V2 vs Prediction")
print("="*70)

# Get first common file
gold_files = {Path(p).stem: str(p) for p in Path(GT_DEST_TTL_V2).glob("*.ttl")}
zero_dir = os.path.join(OUTPUT_ROOT, "vistext_Zeroshot_outputs")
zero_files = {Path(p).stem: str(p) for p in Path(zero_dir).glob("*.ttl")}

common = sorted(set(gold_files.keys()) & set(zero_files.keys()))
sample_id = common[0] if common else None

if sample_id:
    gt_graph = load_ttl_graph(gold_files[sample_id])
    pred_graph = load_ttl_graph(zero_files[sample_id])
    
    gt_points = extract_data_points(gt_graph)
    pred_points = extract_data_points(pred_graph)
    
    gt_dict = {cat: val for cat, val in gt_points}
    pred_dict = {cat: val for cat, val in pred_points}
    
    print(f"\nFile: {sample_id}")
    print(f"Gold has {len(gt_points)} data points")
    print(f"Pred has {len(pred_points)} data points")
    
    print(f"\n{'Category':<40} {'Gold':>12} {'Pred':>12} {'Diff%':>8} {'Match?':>8}")
    print("-" * 84)
    
    all_cats = sorted(set(gt_dict.keys()) | set(pred_dict.keys()))
    for cat in all_cats[:15]:
        gt_val = gt_dict.get(cat)
        pred_val = pred_dict.get(cat)
        
        if gt_val is not None and pred_val is not None:
            if gt_val != 0:
                diff_pct = 100 * abs(gt_val - pred_val) / abs(gt_val)
            else:
                diff_pct = 0 if pred_val == 0 else 100
            match_str = "✓5%" if diff_pct <= 5 else ("~15%" if diff_pct <= 15 else "✗")
            print(f"{cat:<40} {gt_val:>12.1f} {pred_val:>12.1f} {diff_pct:>7.1f}% {match_str:>8}")
        elif gt_val is not None:
            print(f"{cat:<40} {gt_val:>12.1f} {'MISSING':>12} {'-':>8} {'✗':>8}")
        else:
            print(f"{cat:<40} {'MISSING':>12} {pred_val:>12.1f} {'-':>8} {'EXTRA':>8}")
else:
    print("No common files found!")

Sample comparison: Gold V2 vs Prediction

File: 100
Gold has 3 data points
Pred has 20 data points

Category                                         Gold         Pred    Diff%   Match?
------------------------------------------------------------------------------------
*                                            183849.0      MISSING        -        ✗
2000                                          MISSING     245000.0        -    EXTRA
2001                                          MISSING     243000.0        -    EXTRA
2002                                          MISSING     241000.0        -    EXTRA
2003                                          MISSING     238000.0        -    EXTRA
2004                                          MISSING     235000.0        -    EXTRA
2005                                          MISSING     230000.0        -    EXTRA
2006                                          MISSING     225000.0        -    EXTRA
2007                                          MISS

In [40]:
# Diagnostic: Check file distinctness
print("="*70)
print("Diagnostic: File distinctness across methods")
print("="*70)

def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

zero_files = {Path(p).stem: str(p) for p in Path(os.path.join(OUTPUT_ROOT, "vistext_Zeroshot_outputs")).glob('*.ttl')}
one_files = {Path(p).stem: str(p) for p in Path(os.path.join(OUTPUT_ROOT, "vistext_Oneshot_outputs")).glob('*.ttl')}
few_files = {Path(p).stem: str(p) for p in Path(os.path.join(OUTPUT_ROOT, "vistext_Fewshot_outputs")).glob('*.ttl')}

common = sorted(set(zero_files.keys()) & set(one_files.keys()) & set(few_files.keys()))

identical_z_o = identical_z_f = identical_o_f = 0
for img_id in common:
    z_hash = file_hash(zero_files[img_id])
    o_hash = file_hash(one_files[img_id])
    f_hash = file_hash(few_files[img_id])
    
    if z_hash == o_hash: identical_z_o += 1
    if z_hash == f_hash: identical_z_f += 1
    if o_hash == f_hash: identical_o_f += 1

print(f"Common files: {len(common)}")
print(f"Identical (Zero vs One): {identical_z_o}/{len(common)} ({100*identical_z_o/len(common):.1f}%)")
print(f"Identical (Zero vs Few): {identical_z_f}/{len(common)} ({100*identical_z_f/len(common):.1f}%)")
print(f"Identical (One vs Few): {identical_o_f}/{len(common)} ({100*identical_o_f/len(common):.1f}%)")

Diagnostic: File distinctness across methods
Common files: 1000
Identical (Zero vs One): 0/1000 (0.0%)
Identical (Zero vs Few): 0/1000 (0.0%)
Identical (One vs Few): 0/1000 (0.0%)


## Summary

This evaluation uses:
- **Gold TTL V2**: Generated from `datatable` + `caption_L1` only
- **Schema**: `chart:category` and `chart:value` (matching prediction schema)
- **Metrics**: F1 at multiple tolerance levels (5%, 10%, 15%, 20%)